# Onset-envelope transfer functions

The reviewer's feature. Where notebook 6 regresses the EEG on the broadband amplitude envelope,
this regresses it on the envelope's **rising edges**: the envelope low-passed, differentiated and
half-wave rectified, which is the acoustic-onset regressor usually called `peakRate`.

$$x_{\text{onset}}(t) = \max\!\left(\frac{\mathrm{d}}{\mathrm{d}t}\,\text{lowpass}_{f_c}\!\big[x_{\text{env}}(t)\big],\; 0\right)$$

The low-pass cutoff $f_c$ decides what counts as an onset -- a slow swell or a sharp attack --
and there is no principled value for it, so `scripts/trfs/get_onset_transfer_functions.py`
sweeps it from 4 to 20 Hz in 1 Hz steps and saves the whole bank. This notebook pools those over
participants and asks what the cutoff actually changes.

Everything else is notebook 6's: the same Welch windowing, the same estimator with the parts
saved undivided, the same per-participant duration-weighted pooling, the same zero-phase filters
and mains notches, the same average reference.

## Setup

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from mne import pick_info
from mne.io import read_raw_fif
from mne.viz import plot_topomap
from mne.viz.evoked import _get_pos_outlines, _rgb
from mne.viz.topomap import _draw_outlines
from scipy.signal import find_peaks, firwin, freqz, hilbert, iirnotch, kaiserord

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())
sys.path.insert(0, str(ROOT / "src"))

from coherence.pipeline import GAMMATONE_FEATURE_KEY
from trfs.onset_envelope import ONSET_FEATURE_KEY, onset_envelope, resolve_onset_trf_dir
from trfs.transfer_function import rereference_transfer_function, transfer_function_from_parts

# --- what is fitted ----------------------------------------------------------
REFERENCE = "avg"
WINDOW_SECONDS, OVERLAP = 5.0, 0.8
RIDGE = 0.0

ONSET_DIR = resolve_onset_trf_dir(CONFIG, REFERENCE, WINDOW_SECONDS, OVERLAP, ROOT)
EEG_DIR = Path(CONFIG["eeg_dir"])
AUDIO_FEATURES_DIR = Path(CONFIG["audio_features_dir"])
OUT_DIR = ROOT / "figures"
CACHE_ROOT = OUT_DIR / "figure_data"

TEMPLATE_EEG = EEG_DIR / "sub-001_-_ses-001_-_trial-001_-_cond-natural_-_audio01-audiobook_5_1_-_eeg.fif"
RAW_INFO = read_raw_fif(TEMPLATE_EEG, preload=False, verbose="ERROR").pick("eeg").info

paths = sorted(ONSET_DIR.glob("*_-_eeg.npz"))
if not paths:
    raise FileNotFoundError(
        f"no trial files in {ONSET_DIR}\n"
        "run scripts/trfs/get_onset_transfer_functions.py first")

# The cutoff axis, the frequency grid and the transform length are properties of the run.
with np.load(paths[0]) as saved:
    CUTOFFS = saved["cutoffs"]
    frequencies = saved["frequencies"]
    fs = float(saved["fs"])
    nperseg = int(saved["nperseg"])
    FILTER_ORDER = int(saved["filter_order"])
    assert float(saved["window_seconds"]) == WINDOW_SECONDS, "window mismatch with the run"
assert len(frequencies) == nperseg // 2 + 1, "frequency grid does not match the transform"
# Zero lag in the middle of the transform, which is circular with the window's period.
times = (np.arange(nperseg) - nperseg // 2) / fs * 1000

# --- house style, shared with figures 1-6 ------------------------------------
DEFAULT_FONTSIZE, TICK_FONTSIZE, SMALL_FONTSIZE = 10, 8, 6.5
PANEL_LABEL_FONTSIZE = 14
INK, MUTED, HAIRLINE = "#1b1b1a", "#6b6b68", "#d5d5d1"

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": DEFAULT_FONTSIZE,
    "axes.titlesize": DEFAULT_FONTSIZE,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE, "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8, "figure.dpi": 130,
    # TrueType, not matplotlib's default Type 3, which publishers flag.
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def bare(ax):
    """Recessive chrome: drop the top/right rules."""
    ax.spines[["top", "right"]].set_visible(False)
    return ax


print(f"{len(paths)} trials in {ONSET_DIR.name} under {REFERENCE}, "
      f"{sum(p.stat().st_size for p in paths) / 1e9:.1f} GB")
print(f"{len(CUTOFFS)} cutoffs {CUTOFFS[0]:g}-{CUTOFFS[-1]:g} Hz, Butterworth order "
      f"{FILTER_ORDER} applied forwards and backwards")
print(f"{len(frequencies)} bins at {frequencies[1] - frequencies[0]:g} Hz; "
      f"window {nperseg} samples ({nperseg / fs:g} s) at {fs:g} Hz")

## What the feature is

Before any transfer function, the regressor itself. The same three steps the script applies, on
one stimulus: low-pass the broadband envelope, differentiate, keep the positive part.

The cutoff is doing something specific and visible. Low, and the feature marks the slow rise
into each syllable; high, and it starts to resolve structure within one. Neither is obviously
the right answer, which is why the run sweeps it rather than picking.

In [ ]:
# --- options -----------------------------------------------------------------
DEMO_STIMULUS = None            # None -> the first stimulus in the feature directory
DEMO_SECONDS = (10.0, 13.0)     # the stretch drawn
DEMO_CUTOFFS = [4.0, 10.0, 20.0]
FEATURE_FIGSIZE = (6.27, 3.0)
# =============================================================================

demo_path = (AUDIO_FEATURES_DIR / f"{DEMO_STIMULUS}.npz" if DEMO_STIMULUS
             else sorted(AUDIO_FEATURES_DIR.glob("*.npz"))[0])
with np.load(demo_path) as saved:
    demo_envelope = np.asarray(saved[GAMMATONE_FEATURE_KEY], float).reshape(-1)

start, stop = (int(round(second * fs)) for second in DEMO_SECONDS)
window = slice(start, stop)
seconds = np.arange(start, stop) / fs

feature_fig, (ax_envelope, ax_onset) = plt.subplots(
    2, 1, figsize=FEATURE_FIGSIZE, sharex=True, gridspec_kw={"hspace": 0.25})

ax_envelope.plot(seconds, demo_envelope[window], color=INK, lw=0.9,
                 label="Broadband envelope")
ax_envelope.set_ylabel("Envelope")
ax_envelope.legend(fontsize=SMALL_FONTSIZE, frameon=False, loc="upper right")

# Each cutoff on its own maximum: they differ in amplitude by more than they differ in shape,
# and the shape is the point.
colours = plt.get_cmap("viridis")(np.linspace(0.15, 0.85, len(DEMO_CUTOFFS)))
for cutoff, colour in zip(DEMO_CUTOFFS, colours):
    onset = onset_envelope(demo_envelope, fs, cutoff, FILTER_ORDER)[window]
    ax_onset.plot(seconds, onset / onset.max(), color=colour, lw=0.9,
                  label=f"{cutoff:g} Hz")
ax_onset.set_ylabel("Onset envelope\n($\\div$ its maximum)")
ax_onset.set_xlabel("Time (s)")
ax_onset.legend(fontsize=SMALL_FONTSIZE, frameon=False, ncol=len(DEMO_CUTOFFS),
                loc="upper right", title="Low-pass cutoff",
                title_fontsize=SMALL_FONTSIZE)

for ax, label in [(ax_envelope, "a)"), (ax_onset, "b)")]:
    ax.set_xlim(*DEMO_SECONDS)
    bare(ax)
    ax.text(-0.08, 1.02, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = "onset_feature"
feature_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
feature_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

print(f"{demo_path.stem}, {DEMO_SECONDS[0]:g}-{DEMO_SECONDS[1]:g} s")
for cutoff in DEMO_CUTOFFS:
    onset = onset_envelope(demo_envelope, fs, cutoff, FILTER_ORDER)
    duty = float((onset > 0).mean())
    print(f"  {cutoff:4.0f} Hz: non-zero {duty * 100:4.1f}% of samples, "
          f"mean rate {onset.mean():.3g}/s")

## The filters the transfer functions are read through

Notebook 6's design, unchanged: a Kaiser-window FIR evaluated on the fit's own frequency grid
and multiplied onto $H$, with the **magnitude** applied rather than the complex response so it
is zero phase -- latency is the measurement here, and a group delay would bias exactly that.
Every mains harmonic is notched.

One band for every cutoff, rather than each cutoff read through its own. The feature is already
low-passed, so the transfer function above a cutoff is division by a vanishing $S_{xx}$ and
carries nothing; but reading each one through a different band would confound *what the cutoff
does to the response* with *what band it is being looked at in*, which is the comparison this
notebook exists to make.

In [ ]:
# --- options -----------------------------------------------------------------
# The EEG band every cutoff is read through -- the frequency range of the TRF itself, not of
# the regressor. This is the DEFAULT: any figure can ask for another band by argument, since
# applying it is one multiplication along frequency at display time and costs nothing. The one
# place it is not free is the SD divisor in the pooling below, which is computed at this band
# and baked in per trial.
TRF_BAND = (0.5, 30.0)
FILTER_TRANSITION_HZ = 1.0
FILTER_RIPPLE_DB = 60.0
MAINS_HZ, NOTCH_BANDWIDTH_HZ = 50.0, 2.0
STOPBAND_WARN_DB = -40.0
# =============================================================================


def notch_response():
    """Every mains harmonic up to Nyquist, as a product of `iirnotch` magnitudes."""
    response = np.ones_like(frequencies)
    harmonics = np.arange(MAINS_HZ, fs / 2, MAINS_HZ)
    for harmonic in harmonics:
        b, a = iirnotch(harmonic, harmonic / NOTCH_BANDWIDTH_HZ, fs=fs)
        response *= np.abs(freqz(b, a, worN=frequencies, fs=fs)[1])
    return response, harmonics


def band_filter(low, high, transition=FILTER_TRANSITION_HZ):
    """`(magnitude on the fit's grid, taps used, taps wanted)` for a Kaiser FIR bandpass."""
    wanted, beta = kaiserord(FILTER_RIPPLE_DB, transition / (fs / 2))
    used = min(wanted, nperseg - 1) | 1          # odd -> type I, exactly linear phase
    taps = firwin(used, [low, high], window=("kaiser", beta), pass_zero=False, fs=fs)
    return np.abs(freqz(taps, worN=frequencies, fs=fs)[1]), used, wanted


def filter_quality(response, low, high, transition=FILTER_TRANSITION_HZ):
    """`(worst stopband dB, where, DC dB)` -- what degrades when the taps run out."""
    decibels = 20 * np.log10(response + 1e-15)
    stop = (frequencies < low - transition) | (frequencies > high + transition)
    worst = int(np.argmax(decibels[stop]))
    return decibels[stop][worst], frequencies[stop][worst], decibels[0]


NOTCHES, HARMONICS = notch_response()

_band_responses: dict[tuple, np.ndarray] = {}


def band_response(low, high, transition=FILTER_TRANSITION_HZ):
    """The full display response -- bandpass times the mains notches -- for one band.

    Cached, because a figure that sweeps something else may ask for the same band once per
    cutoff and the Kaiser design is the expensive part of it.
    """
    key = (float(low), float(high), float(transition))
    if key not in _band_responses:
        _band_responses[key] = band_filter(low, high, transition)[0] * NOTCHES
    return _band_responses[key]


BANDPASS, TAPS_USED, TAPS_WANTED = band_filter(*TRF_BAND)
TRF_RESPONSE = band_response(*TRF_BAND)
WORST_STOPBAND, WORST_AT, DC_DB = filter_quality(BANDPASS, *TRF_BAND)

print(f"Kaiser FIR {TRF_BAND[0]:g}-{TRF_BAND[1]:g} Hz: {TAPS_USED}/{TAPS_WANTED} taps "
      f"({TAPS_USED / fs:.2f} s kernel, window is {nperseg / fs:g} s)")
print(f"  stopband {WORST_STOPBAND:+.1f} dB at {WORST_AT:.1f} Hz, DC {DC_DB:+.1f} dB"
      + ("   <- LEAKY" if WORST_STOPBAND > STOPBAND_WARN_DB else ""))
print(f"  {len(HARMONICS)} mains harmonics notched, {NOTCH_BANDWIDTH_HZ:g} Hz wide")

## Pooling

One pass over the trial files. Each trial fitted, duration-weighted within a participant, then
averaged over participants -- with the two stages collapsed into one accumulator per cutoff,
which is what makes 17 cutoffs cost one pass rather than 17. With every participant's total
weight known in advance the two are the same sum:

$$\frac{1}{S}\sum_s \frac{\sum_t w_t X_t}{W_s} \;=\; \sum_{s,t} \frac{1}{S}\frac{w_t}{W_s} X_t$$

Each channel is divided by the standard deviation of the EEG the display filter passes, so a
topography shows where the response is rather than where the EEG is large. The same divisor for
every cutoff, for the same reason the band is shared.

Cached per reference, window, ridge, band **and trial count** -- the last so a cache written
while the run was still going cannot be mistaken for a complete one.

In [ ]:
def cache_path():
    key = json.dumps([REFERENCE, WINDOW_SECONDS, OVERLAP, RIDGE, TRF_BAND, list(CUTOFFS),
                      FILTER_RIPPLE_DB, FILTER_TRANSITION_HZ, MAINS_HZ, NOTCH_BANDWIDTH_HZ,
                      len(paths), 3], sort_keys=True)
    directory = CACHE_ROOT / REFERENCE
    directory.mkdir(parents=True, exist_ok=True)
    return directory / f"onset_trf_{hashlib.sha1(key.encode()).hexdigest()[:12]}.npz"


def band_standard_deviation(psd, response):
    """Each channel's SD of the EEG `response` passes, from its one-sided PSD.

    By Parseval the variance is the integral of the PSD, so weighting the integrand by the
    filter's POWER response gives the SD of the filtered signal exactly -- band edges, roll-off
    and mains notches included.
    """
    return np.sqrt((psd * (response ** 2)[:, None]).sum(0) * (frequencies[1] - frequencies[0]))


def pool_cutoffs():
    """`(H, S_xx, channels, n_subjects, hours)`; H is (n_cutoffs, F, n_channels).

    The stimulus PSD is pooled too, on the same weights. It is what makes a magnitude
    comparable between cutoffs: `H = S_xy / S_xx` is homogeneous of degree -1 in the feature,
    so a feature that grows with cutoff shrinks `|H|` for reasons that have nothing to do with
    the response. `sqrt(sum S_xx df)` is that feature's SD, and multiplying it back in gives the
    standardised coefficient -- what the regression would return on a z-scored regressor.
    """
    weights: dict[str, float] = {}
    for path in paths:
        with np.load(path) as saved:
            weights[str(saved["subject"])] = (weights.get(str(saved["subject"]), 0.0)
                                              + float(saved["analysed_duration"]))
    scale = 1.0 / len(weights)
    hours = sum(weights.values()) / 3600

    # Two pools from the SAME pass: with the per-channel EEG SD divided out and without. The
    # divisor is per trial, so it cannot be undone afterwards -- a magnitude read off the
    # normalised fit is in units of "per EEG standard deviation", which is the right thing for
    # a topography and the wrong thing for a response amplitude.
    channels, totals, raw_totals = None, None, None
    stimulus_totals = np.zeros((len(CUTOFFS), len(frequencies)))
    for count, path in enumerate(paths, 1):
        with np.load(path) as saved:
            cross = saved[f"cross_spectrum_{ONSET_FEATURE_KEY}"]      # (F, n_cutoffs, Cy)
            stimulus_psd = saved[f"stimulus_psd_{ONSET_FEATURE_KEY}"]  # (F, n_cutoffs)
            names = [str(name) for name in saved["channel_names"]]
            psd = saved["eeg_psd"]
            subject = str(saved["subject"])
            duration = float(saved["analysed_duration"])
            assert np.array_equal(saved["cutoffs"], CUTOFFS), f"{path.stem}: different cutoffs"

        # The cross-spectrum is linear in the EEG, so it moves to any reference the file's
        # channels can define; the PSD cannot, which is why it is read under the fit's own.
        cross, trial_channels = rereference_transfer_function(cross, names, REFERENCE)
        if channels is None:
            channels = list(trial_channels)
            totals = np.zeros((len(CUTOFFS), len(frequencies), len(channels)), complex)
            raw_totals = np.zeros_like(totals)
        # One channel axis throughout is what makes the collapsed sum equal the two-stage one.
        assert trial_channels == channels, f"{path.stem}: unexpected channel axis"

        keep = [names.index(name) for name in trial_channels]
        sd = band_standard_deviation(psd[:, keep], TRF_RESPONSE)
        # `transfer_function_from_parts` broadcasts the stimulus axis against the EEG one, so
        # this divides all 17 cutoffs at once: (F, n_cutoffs, Cy) / (F, n_cutoffs, 1).
        H = transfer_function_from_parts(cross, stimulus_psd, RIDGE)
        share = scale * duration / weights[subject]
        totals += share * (H / sd).transpose(1, 0, 2)
        raw_totals += share * H.transpose(1, 0, 2)
        stimulus_totals += share * stimulus_psd.T
        if count % 100 == 0 or count == len(paths):
            print(f"  {count}/{len(paths)} trials", flush=True)

    return totals, raw_totals, stimulus_totals, channels, len(weights), hours


cache = cache_path()
if cache.exists():
    with np.load(cache) as saved:
        ONSET_FITS = saved["fits"]
        ONSET_FITS_RAW = saved["fits_raw"]
        ONSET_STIMULUS_PSD = saved["stimulus_psd"]
        channels = [str(name) for name in saved["channel_names"]]
        n_subjects = int(saved["n_subjects"])
        listening_hours = float(saved["hours"])
    print(f"cached: {cache.name}")
else:
    (ONSET_FITS, ONSET_FITS_RAW, ONSET_STIMULUS_PSD, channels,
     n_subjects, listening_hours) = pool_cutoffs()
    np.savez_compressed(cache, fits=ONSET_FITS, fits_raw=ONSET_FITS_RAW,
                        stimulus_psd=ONSET_STIMULUS_PSD,
                        channel_names=np.array(channels),
                        n_subjects=n_subjects, hours=listening_hours)
    print(f"written: {cache.name} ({cache.stat().st_size / 1e6:.0f} MB)")

INFO = pick_info(RAW_INFO, [RAW_INFO.ch_names.index(name) for name in channels])
# Position-to-colour: nearby channels get nearby colours, so a butterfly shows *where* a
# deflection is without needing 64 legend entries.
CHANNEL_COLOURS = _rgb(*np.array([channel["loc"][:3] for channel in INFO["chs"]]).T)


def plot_sensor_key(ax):
    """The head that says which colour is where."""
    positions, outlines = _get_pos_outlines(INFO, np.arange(len(INFO.ch_names)),
                                            sphere=(0, 0, 0, 0.095))
    _draw_outlines(ax, outlines)
    ax.scatter(*positions.T, c=CHANNEL_COLOURS, s=7, linewidths=0)
    ax.set_aspect("equal")
    ax.set_axis_off()


def onset_response(index, band=None, normalised=True):
    """Cutoff `index` in the lag domain: `(trf, envelope)`, both (n_lags, n_channels).

    `band` is the EEG band to read through, `(low, high)` in Hz, defaulting to `TRF_BAND`. It
    is applied here rather than in the pooling, so asking for another band is free -- but note
    that `normalised=True` divides by an SD computed at `TRF_BAND`, so a band far from that one
    is being normalised by a different band's amplitude.

    `normalised=False` returns the fit before the per-channel EEG SD was divided out -- the
    response in the EEG's own units rather than per standard deviation.
    """
    fits = ONSET_FITS if normalised else ONSET_FITS_RAW
    response = TRF_RESPONSE if band is None else band_response(*band)
    spectrum = fits[index] * response[:, None]
    trf = np.roll(np.fft.irfft(spectrum, n=nperseg, axis=0), nperseg // 2, axis=0)
    return trf, np.abs(hilbert(trf, axis=0))


# The feature's own SD at each cutoff, by Parseval. Printed because it is the confound in any
# magnitude comparison across cutoffs, and it is large.
FEATURE_SD = np.sqrt(ONSET_STIMULUS_PSD.sum(1) * (frequencies[1] - frequencies[0]))

print(f"{REFERENCE}: {n_subjects} participants, {len(channels)} channels, "
      f"{listening_hours:.1f} listening hours; ridge {RIDGE:g}, "
      f"{TRF_BAND[0]:g}-{TRF_BAND[1]:g} Hz SD")
print(f"feature SD rises {FEATURE_SD[0]:.3g} -> {FEATURE_SD[-1]:.3g} "
      f"({FEATURE_SD[-1] / FEATURE_SD[0]:.2f}x) from {CUTOFFS[0]:g} to {CUTOFFS[-1]:g} Hz -- "
      "|H| falls by that factor for no neural reason")

## Figure 10 -- the onset transfer function, and what the cutoff does to it

**(a)** every channel at one cutoff, with the mean over a fronto-central cluster in black. The
cluster mean is what makes the peak structure legible: 64 traces of opposite polarity cancel to
something shapeless, and the response to an acoustic onset is fronto-central, so averaging the
channels that carry it recovers the sequence. The rules and the topographies below are placed at
**its** extrema, found rather than chosen.

**(b)** that same cluster mean at every cutoff, signed, on a diverging scale. Signed rather than
its envelope, because the thing to see is whether the alternation survives the cutoff -- an
envelope would fold the polarity away and show only that *something* is there.

**(d)** how big the response is at each cutoff, twice. `H = S_xy / S_xx` is homogeneous of
degree $-1$ in the feature, so a regressor whose amplitude grows with cutoff shrinks $|H|$ by
exactly that factor whatever the brain is doing -- and the onset feature's SD rises 3.3x from
4 to 20 Hz. The grey curve is $|H|$ as fitted; the red one multiplies the feature's own SD back
in, which is the coefficient a z-scored regressor would have given. **The red curve is the one
to read as a response**; the grey one is mostly the regressor rescaling itself.

**(c)** each component's latency against cutoff. If the cutoff only rescaled the regressor this
would be flat: $H = S_{xy}/S_{xx}$ is homogeneous of degree $-1$ in the feature, so a constant
factor cancels from it exactly. Any slope is the cutoff changing *which* acoustic events are
marked.

In [ ]:
# ===================== figure 10 options =====================================
ONSET_FIGSIZE = (6.27, 7.9)
BUTTERFLY_CUTOFF = 30.0         # which cutoff panel (a) draws
ONSET_BAND = None               # EEG band to read through; None uses TRF_BAND
ONSET_LAGS = (-1000.0, 2000.0)
ONSET_XTICKS = None             # None derives round ticks from ONSET_LAGS; a list overrides
ONSET_HEADROOM = 1.12           # the maps sit under the panel now, so the traces can fill it

# The response to an acoustic onset is fronto-central, so this is where its peaks are defined.
ONSET_CLUSTER = ["F1", "Fz", "F2", "FC1", "FCz", "FC2", "C1", "Cz", "C2"]
# Peaks of the cluster mean: `None` finds them, a list of lags in ms pins them by hand.
ONSET_PEAK_LAGS = None
N_ONSET_PEAKS = 4
PEAK_SEARCH = (-60.0, 250.0)    # where components are looked for
PEAK_MIN_HEIGHT = 0.12          # of the largest excursion, to reject ripple
PEAK_MIN_SEPARATION_MS = 25.0   # half a period of the 20 Hz top edge

TOPO_WIDTH = 3                  # adjacent samples averaged at each peak
IMAGE_CMAP = "RdBu_r"
TRACK_WINDOW_MS = 35.0          # how far a component may move between neighbouring cutoffs
# Panel (d). Read off the fit BEFORE the per-channel EEG SD was divided out: that divisor
# turns a magnitude into "per EEG standard deviation", which is what a topography wants and not
# what a response amplitude does. Set True to read the normalised fit instead.
MAGNITUDE_SD_NORMALISED = False
# The two lines: as fitted, and with the feature's own scale put back.
RAW_COLOUR, STANDARDISED_COLOUR = "#6b6b68", "#e2431f"
# =============================================================================

def round_ticks(low, high, target=6):
    """Round, evenly spaced ticks spanning `low`..`high`, about `target` of them."""
    step = 10.0 ** np.floor(np.log10((high - low) / target))
    for multiple in (1, 2, 2.5, 5, 10):
        if (high - low) / (step * multiple) <= target:
            step *= multiple
            break
    return np.arange(np.ceil(low / step) * step, high + step / 2, step)


onset_xticks = round_ticks(*ONSET_LAGS) if ONSET_XTICKS is None else ONSET_XTICKS

butterfly_index = int(np.argmin(np.abs(CUTOFFS - BUTTERFLY_CUTOFF)))
if not np.isclose(CUTOFFS[butterfly_index], BUTTERFLY_CUTOFF):
    print(f"note: {BUTTERFLY_CUTOFF:g} Hz is not in the run; drawing the nearest cutoff, "
          f"{CUTOFFS[butterfly_index]:g} Hz")
cluster = [INFO.ch_names.index(name) for name in ONSET_CLUSTER if name in INFO.ch_names]
assert cluster, "none of ONSET_CLUSTER is in this reference's montage"

shown = (times >= ONSET_LAGS[0]) & (times <= ONSET_LAGS[1])
searched = (times >= PEAK_SEARCH[0]) & (times <= PEAK_SEARCH[1])
# Signed, not the envelope: the alternation is the structure, and an envelope hides it.
cluster_means = np.array([onset_response(index, ONSET_BAND)[0][:, cluster].mean(-1)
                          for index in range(len(CUTOFFS))])


def cluster_peaks(trace, count=N_ONSET_PEAKS):
    """The `count` largest extrema of `trace` inside the search window, in time order.

    Run on the absolute value so a trough counts as much as a peak -- the sequence alternates,
    and taking only maxima would return every other component.
    """
    magnitude = np.where(searched, np.abs(trace), 0.0)
    found, _ = find_peaks(magnitude, distance=max(1, int(PEAK_MIN_SEPARATION_MS / 1000 * fs)),
                          height=PEAK_MIN_HEIGHT * magnitude.max())
    return np.sort(found[np.argsort(magnitude[found])[::-1][:count]])


reference_peaks = (cluster_peaks(cluster_means[butterfly_index]) if ONSET_PEAK_LAGS is None
                   else np.array([int(np.argmin(np.abs(times - lag)))
                                  for lag in ONSET_PEAK_LAGS]))
peak_lags = times[reference_peaks]

# Follow each component across cutoffs: the extremum of the same sign nearest to where it sat
# at the neighbouring cutoff. Tracking from neighbour to neighbour rather than from one
# reference row lets a component drift a long way overall without ever jumping between rows.
tracked = np.full((len(CUTOFFS), len(reference_peaks)), np.nan)
span = int(round(TRACK_WINDOW_MS / 1000 * fs))
for column, start in enumerate(reference_peaks):
    sign = np.sign(cluster_means[butterfly_index, start])
    for direction in (1, -1):
        here = start
        rows = (range(butterfly_index, len(CUTOFFS)) if direction > 0
                else range(butterfly_index, -1, -1))
        for row in rows:
            window = slice(max(here - span, 0), here + span + 1)
            candidate = int(np.argmax(sign * cluster_means[row, window])) + window.start
            tracked[row, column] = times[candidate]
            here = candidate

onset_fig = plt.figure(figsize=ONSET_FIGSIZE)
outer = onset_fig.add_gridspec(3, 1, height_ratios=[2.5, 2.1, 1.6], hspace=0.50)
top = outer[0].subgridspec(2, 1, height_ratios=[3, 1.15], hspace=0.50)
# (c) and (d) share the bottom row: both are read against the cutoff, and side by side the
# same x axis carries them.
bottom = outer[2].subgridspec(1, 2, wspace=0.55)
ax_trf = onset_fig.add_subplot(top[0])

# --- (a) the butterfly, with the cluster mean over it ------------------------
trf, envelope = onset_response(butterfly_index, ONSET_BAND)
for index in range(trf.shape[1]):
    ax_trf.plot(times[shown], trf[shown, index], color=CHANNEL_COLOURS[index],
                lw=0.5, alpha=0.45, zorder=2)
ax_trf.plot(times[shown], cluster_means[butterfly_index][shown], color="k", lw=1.6, zorder=5,
            label=f"Fronto-central mean ({len(cluster)} channels)")
limit = float(np.abs(trf[shown]).max()) * ONSET_HEADROOM
ax_trf.set_ylim(-limit, limit)
ax_trf.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
ax_trf.legend(fontsize=SMALL_FONTSIZE, frameon=False, loc="lower right")
ax_trf.set_ylabel("Activity (a.u.)")
onset_band = TRF_BAND if ONSET_BAND is None else ONSET_BAND
ax_trf.set_title(f"Onset TRF, {onset_band[0]:g}-{onset_band[1]:g} Hz EEG "
                 f"({CUTOFFS[butterfly_index]:g} Hz peakRate cutoff)",
                 fontsize=DEFAULT_FONTSIZE)
plot_sensor_key(ax_trf.inset_axes([0.0, 0.62, 0.11, 0.34]))

# --- the topographies, in a strip rather than over the traces ----------------
strip = top[1].subgridspec(1, len(reference_peaks), wspace=0.08)
for column, sample in enumerate(reference_peaks):
    start = min(max(sample - TOPO_WIDTH // 2, 0), len(times) - TOPO_WIDTH)
    values = trf[start:start + TOPO_WIDTH].mean(0)
    extreme = float(np.abs(values).max())
    ax_trf.axvline(times[sample], color="k", lw=0.7, ls="--", alpha=0.7, zorder=1)
    topo_ax = onset_fig.add_subplot(strip[column])
    plot_topomap(values, INFO, axes=topo_ax, show=False, cmap="RdBu_r",
                 vlim=(-extreme, extreme), contours=0, sensors=False)
    topo_ax.set_title(f"{times[sample]:.0f} ms", fontsize=SMALL_FONTSIZE, pad=2)

# --- (b) the cluster mean at every cutoff ------------------------------------
ax_image = onset_fig.add_subplot(outer[1], sharex=ax_trf)
plate = cluster_means[:, shown]
# Each row on its own maximum, and symmetric about zero: the feature's amplitude falls with
# cutoff, and a shared scale would draw that rather than the structure.
plate = plate / np.abs(plate).max(axis=1, keepdims=True)
mesh = ax_image.pcolormesh(times[shown], CUTOFFS, plate, cmap=IMAGE_CMAP, shading="nearest",
                           vmin=-1, vmax=1, rasterized=True)
for column in range(tracked.shape[1]):
    ax_image.plot(tracked[:, column], CUTOFFS, color="k", lw=0.9, ls="--", zorder=3)
ax_image.set_ylabel("Low-pass cutoff (Hz)")
ax_image.set_title("Fronto-central mean, every cutoff", fontsize=DEFAULT_FONTSIZE)
colorbar = plt.colorbar(mesh, ax=ax_image, pad=0.02)
colorbar.set_ticks([-1, 0, 1])
colorbar.set_label("$\\div$ row's largest", fontsize=SMALL_FONTSIZE)
colorbar.ax.tick_params(labelsize=TICK_FONTSIZE)

# --- (c) each component's latency against cutoff -----------------------------
ax_latency = onset_fig.add_subplot(bottom[0])
colours = plt.get_cmap("viridis")(np.linspace(0.1, 0.8, tracked.shape[1]))
# Labelled at the end of each line rather than in a legend: at four components in a panel
# this narrow the legend box sat on top of the lines it was describing.
cutoff_span = CUTOFFS[-1] - CUTOFFS[0]
for column, colour in enumerate(colours):
    gradient = np.polyfit(CUTOFFS, tracked[:, column], 1)[0]
    ax_latency.plot(CUTOFFS, tracked[:, column], marker="o", ms=3.0, lw=1.0, color=colour)
    ax_latency.annotate(f"{gradient:+.2f}", (CUTOFFS[-1], tracked[-1, column]),
                        textcoords="offset points", xytext=(4, 0), fontsize=SMALL_FONTSIZE,
                        color=colour, va="center", ha="left")
ax_latency.set_xlim(CUTOFFS[0] - 0.05 * cutoff_span, CUTOFFS[-1] + 0.30 * cutoff_span)
ax_latency.set_xlabel("Low-pass cutoff (Hz)")
ax_latency.set_ylabel("Latency (ms)")
ax_latency.set_xticks(CUTOFFS[::4])
ax_latency.set_title("Component latency", fontsize=DEFAULT_FONTSIZE)
ax_latency.text(0.02, 0.03, "ms/Hz at each line's end", transform=ax_latency.transAxes,
                fontsize=SMALL_FONTSIZE, color=MUTED, va="bottom", ha="left")

# --- (d) how big the response is, against cutoff -----------------------------
# The peak of the analytic envelope of the cluster mean: one number per cutoff, and free of
# the polarity alternation that would make a signed maximum jump between components.
magnitude_means = np.array([
    onset_response(index, ONSET_BAND, MAGNITUDE_SD_NORMALISED)[0][:, cluster].mean(-1)
    for index in range(len(CUTOFFS))])
peak_magnitude = np.array([np.abs(hilbert(trace))[searched].max()
                           for trace in magnitude_means])

ax_magnitude = onset_fig.add_subplot(bottom[1])
ax_magnitude.plot(CUTOFFS, peak_magnitude, marker="o", ms=3.0, lw=1.2, color=RAW_COLOUR,
                  label="$|H|$ as fitted")
ax_magnitude.set_xlabel("Low-pass cutoff (Hz)")
ax_magnitude.set_ylabel("Peak envelope (a.u.)", color=RAW_COLOUR)
ax_magnitude.set_title("Response magnitude"
                       + ("" if MAGNITUDE_SD_NORMALISED else "\n(no EEG SD normalisation)"),
                       fontsize=DEFAULT_FONTSIZE)
ax_magnitude.tick_params(axis="y", colors=RAW_COLOUR)
ax_magnitude.set_xticks(CUTOFFS[::4])

# The same curve with the feature's own scale put back. `H = S_xy / S_xx` is homogeneous of
# degree -1 in the feature, so a regressor that grows with cutoff shrinks |H| by exactly that
# factor whatever the brain does; multiplying the SD back in is the coefficient a z-scored
# regressor would have given, and is the one to read as a response.
standardised = ax_magnitude.twinx()
standardised.plot(CUTOFFS, peak_magnitude * FEATURE_SD, marker="s", ms=3.0, lw=1.2,
                  color=STANDARDISED_COLOUR, label=r"$|H| \times \mathrm{SD}(x)$")
standardised.set_ylabel("Standardised", color=STANDARDISED_COLOUR)
standardised.tick_params(axis="y", colors=STANDARDISED_COLOUR)
standardised.spines[["top"]].set_visible(False)

handles = ax_magnitude.get_legend_handles_labels()[0] + \
    standardised.get_legend_handles_labels()[0]
ax_magnitude.legend(handles=handles, fontsize=SMALL_FONTSIZE, frameon=False,
                    loc="upper center")

for ax in (ax_trf, ax_image):
    ax.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
    ax.set_xlim(*ONSET_LAGS)
    ax.set_xticks(onset_xticks)
ax_image.set_xlabel("Lag (ms)")
for ax in (ax_trf, ax_latency):
    bare(ax)

for ax, label, offset in [(ax_trf, "a)", -0.09), (ax_image, "b)", -0.09),
                          (ax_latency, "c)", -0.30), (ax_magnitude, "d)", -0.30)]:
    ax.text(offset, 1.03, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = f"onset_trf_{REFERENCE}"
onset_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
onset_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

print(f"fronto-central cluster: {', '.join(INFO.ch_names[i] for i in cluster)}")
print(f"components at the {CUTOFFS[butterfly_index]:g} Hz cutoff: "
      + ", ".join(f"{lag:+.0f} ms ({'+' if cluster_means[butterfly_index][s] > 0 else '-'})"
                  for lag, s in zip(peak_lags, reference_peaks)))
print(f"\n{'component':>12s} {'4 Hz':>8s} {'20 Hz':>8s} {'drift':>9s} {'slope':>11s}")
for column in range(tracked.shape[1]):
    gradient = np.polyfit(CUTOFFS, tracked[:, column], 1)[0]
    print(f"{peak_lags[column]:9.0f} ms {tracked[0, column]:7.0f}  {tracked[-1, column]:7.0f}  "
          f"{tracked[-1, column] - tracked[0, column]:+8.0f}  {gradient:+8.2f} ms/Hz")

# Does the EEG SD normalisation change the PROFILE, or only its units? The divisor does not
# depend on the cutoff, so it can only act through the per-trial reweighting it causes.
other = np.array([
    np.abs(hilbert(onset_response(index, ONSET_BAND, not MAGNITUDE_SD_NORMALISED)[0]
                   [:, cluster].mean(-1)))[searched].max()
    for index in range(len(CUTOFFS))])
shape = np.corrcoef(peak_magnitude / peak_magnitude.max(), other / other.max())[0, 1]
print(f"\nprofile with vs without the EEG SD normalisation: r = {shape:.4f}; "
      f"peak at {CUTOFFS[np.argmax(peak_magnitude)]:g} Hz either way"
      if np.argmax(peak_magnitude) == np.argmax(other) else
      f"\nprofile with vs without the EEG SD normalisation: r = {shape:.4f}; peaks at "
      f"{CUTOFFS[np.argmax(peak_magnitude)]:g} and {CUTOFFS[np.argmax(other)]:g} Hz")

print(f"\n{'cutoff':>7s} {'feature SD':>11s} {'|H| peak':>11s} {'standardised':>13s}")
for index, cutoff in enumerate(CUTOFFS):
    print(f"{cutoff:5.0f} Hz {FEATURE_SD[index]:11.3g} {peak_magnitude[index]:11.3g} "
          f"{peak_magnitude[index] * FEATURE_SD[index]:13.3g}")
best_raw, best_std = int(np.argmax(peak_magnitude)), int(np.argmax(peak_magnitude * FEATURE_SD))
print(f"largest |H| at {CUTOFFS[best_raw]:g} Hz; largest standardised response at "
      f"{CUTOFFS[best_std]:g} Hz")
print("The two differ because the feature's SD rises "
      f"{FEATURE_SD[-1] / FEATURE_SD[0]:.2f}x over this range; the standardised curve is the "
      "one that\nreflects the response rather than the regressor's units.")

### The onset TRF on its own

Figure 10's panel (a) as a figure in its own right: one cutoff, every channel, the fronto-central
mean over the top, and a topography at each of its peaks. Nothing about the cutoff sweep -- this
is the response itself, for a talk slide or a single-panel figure.

`SINGLE_CUTOFF` picks the cutoff; everything else is inherited from the figure above so the two
cannot show different things, and any of it can be overridden in the block at the top.

In [ ]:
# ===================== single-cutoff options =================================
# The EEG band the response is read through -- the frequency range of the TRF itself.
# Free to change: one multiplication along frequency at display time, no re-pooling.
SINGLE_BAND = (1, 30.0)
# The peakRate low-pass cutoff, a property of the REGRESSOR, which the run had to sweep.
SINGLE_CUTOFF = 10.0            # must be one the run swept -- CUTOFFS lists them
SINGLE_FIGSIZE = (6.27, 3.4)
SINGLE_LAGS = (-1000.0, 2000.0)
SINGLE_XTICKS = None            # None derives round ticks from SINGLE_LAGS
SINGLE_HEADROOM = 1.12
SINGLE_N_PEAKS = N_ONSET_PEAKS  # inherited; set a number to override
SINGLE_PEAK_LAGS = [59, 110, 177, 450]         # None finds them, a list of lags in ms pins them
SINGLE_NORMALISED = True        # divided by the per-channel EEG SD, as figure 10's (a) is
SINGLE_SHOW_MEAN = False        # the fronto-central mean over the butterfly
SINGLE_KEY_RECT = [0.0, 0.62, 0.11, 0.34]
# =============================================================================

if not np.any(np.isclose(CUTOFFS, SINGLE_CUTOFF)):
    raise ValueError(
        f"{SINGLE_CUTOFF:g} Hz is not in this run, which swept "
        f"{', '.join(f'{c:g}' for c in CUTOFFS)} Hz.\n"
        "Re-run scripts/trfs/get_onset_transfer_functions.py with a --cutoff-range that "
        f"includes it, e.g. --cutoff-range {CUTOFFS[0]:g} {SINGLE_CUTOFF:g} 1")
single_index = int(np.argmin(np.abs(CUTOFFS - SINGLE_CUTOFF)))

single_trf = onset_response(single_index, SINGLE_BAND, SINGLE_NORMALISED)[0]
single_mean = single_trf[:, cluster].mean(-1)
single_shown = (times >= SINGLE_LAGS[0]) & (times <= SINGLE_LAGS[1])
single_ticks = (round_ticks(*SINGLE_LAGS) if SINGLE_XTICKS is None else SINGLE_XTICKS)

# The same finder figure 10 uses, on this cutoff's own cluster mean.
if SINGLE_PEAK_LAGS is None:
    single_peaks = cluster_peaks(single_mean, SINGLE_N_PEAKS)
else:
    single_peaks = np.array([int(np.argmin(np.abs(times - lag))) for lag in SINGLE_PEAK_LAGS])

single_fig = plt.figure(figsize=SINGLE_FIGSIZE)
single_grid = single_fig.add_gridspec(2, 1, height_ratios=[3, 1.15], hspace=0.50)
ax_single = single_fig.add_subplot(single_grid[0])

for index in range(single_trf.shape[1]):
    ax_single.plot(times[single_shown], single_trf[single_shown, index],
                   color=CHANNEL_COLOURS[index], lw=0.5, alpha=0.45, zorder=2)
if SINGLE_SHOW_MEAN:
    ax_single.plot(times[single_shown], single_mean[single_shown], color="k", lw=1.6, zorder=5,
                   label=f"Fronto-central mean ({len(cluster)} channels)")

single_limit = float(np.abs(single_trf[single_shown]).max()) * SINGLE_HEADROOM
ax_single.set_ylim(-single_limit, single_limit)
ax_single.set_xlim(*SINGLE_LAGS)
ax_single.set_xticks(single_ticks)
ax_single.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
ax_single.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
ax_single.set_xlabel("Lag (ms)")
ax_single.set_ylabel("Activity (a.u.)")
ax_single.set_title(f"Onset TRF, {SINGLE_BAND[0]:g}-{SINGLE_BAND[1]:g} Hz EEG "
                    f"({CUTOFFS[single_index]:g} Hz peakRate cutoff)",
                    fontsize=DEFAULT_FONTSIZE)
# Only when there is something to label: the peaks are still found on the cluster mean
# whether or not it is drawn, so the rules and the maps are unaffected by the switch.
if SINGLE_SHOW_MEAN:
    ax_single.legend(fontsize=SMALL_FONTSIZE, frameon=False, loc="lower right")
bare(ax_single)
plot_sensor_key(ax_single.inset_axes(SINGLE_KEY_RECT))

# The maps in a strip below rather than over the traces, so the y limits can stay tight.
single_strip = single_grid[1].subgridspec(1, len(single_peaks), wspace=0.08)
for column, sample in enumerate(single_peaks):
    start = min(max(sample - TOPO_WIDTH // 2, 0), len(times) - TOPO_WIDTH)
    values = single_trf[start:start + TOPO_WIDTH].mean(0)
    extreme = float(np.abs(values).max())
    ax_single.axvline(times[sample], color="k", lw=0.7, ls="--", alpha=0.7, zorder=1)
    topo_ax = single_fig.add_subplot(single_strip[column])
    plot_topomap(values, INFO, axes=topo_ax, show=False, cmap="RdBu_r",
                 vlim=(-extreme, extreme), contours=0, sensors=False)
    topo_ax.set_title(f"{times[sample]:.0f} ms", fontsize=SMALL_FONTSIZE, pad=2)

stem = (f"onset_trf_{SINGLE_BAND[0]:g}-{SINGLE_BAND[1]:g}Hz"
        f"_cut{CUTOFFS[single_index]:g}_{REFERENCE}")
single_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
single_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

normalisation = "per EEG SD" if SINGLE_NORMALISED else "no EEG SD normalisation"
print(f"{SINGLE_BAND[0]:g}-{SINGLE_BAND[1]:g} Hz EEG, {CUTOFFS[single_index]:g} Hz peakRate "
      f"cutoff, {normalisation}, {n_subjects} participants")
if SINGLE_NORMALISED and tuple(SINGLE_BAND) != tuple(TRF_BAND):
    print(f"  note: the EEG SD divisor was computed at {TRF_BAND[0]:g}-{TRF_BAND[1]:g} Hz, not "
          f"at this band -- change TRF_BAND and re-pool to match, or set "
          "SINGLE_NORMALISED=False")
for sample in single_peaks:
    values = single_trf[sample]
    strongest = INFO.ch_names[int(np.argmax(np.abs(values)))]
    print(f"  {times[sample]:6.0f} ms  {'+' if single_mean[sample] > 0 else '-'}  "
          f"strongest {strongest}  ({np.abs(values).max():.3g})")

## Per stimulus

The reviewer's request: the same fit, pooled within each stimulus instead of over everyone. Each
stimulus is the duration-weighted mean of the trials that heard it, the same estimator and the same
per-channel EEG SD divisor as above, at one cutoff, so a row is directly comparable with the pooled
response.

By default the heatmap shows each stimulus's **channel-averaged envelope**: the Hilbert envelope of
every channel's TRF, averaged over the 64. Taking the envelope before averaging matters here. The
signed channels cancel, because an average reference sums them to zero, but their envelopes do
not. `PER_STIMULUS_MEASURE = "channel"` shows one channel's signed TRF instead,
`PER_STIMULUS_CHANNEL` (FCz by default), on a diverging scale. One channel does not cancel, so
there is no reason to fold its polarity away.

The number of listeners per stimulus is very uneven, from 1 to 72. A row heard by one person is one
trial's fit and is far noisier than one heard by many, so the count is drawn next to each row and
`PER_STIMULUS_MIN_TRIALS` can drop the thin ones.

In [ ]:
# ===================== per-stimulus options ==================================
# One cutoff and one band, inherited from the single-cutoff figure so the two show the same fit.
PER_STIMULUS_CUTOFF = SINGLE_CUTOFF
PER_STIMULUS_BAND = SINGLE_BAND
# =============================================================================

if not np.any(np.isclose(CUTOFFS, PER_STIMULUS_CUTOFF)):
    raise ValueError(f"{PER_STIMULUS_CUTOFF:g} Hz is not in this run; CUTOFFS lists those it is")
stimulus_cutoff_index = int(np.argmin(np.abs(CUTOFFS - PER_STIMULUS_CUTOFF)))


def stimulus_cache_path():
    key = json.dumps([REFERENCE, WINDOW_SECONDS, OVERLAP, RIDGE, TRF_BAND,
                      float(CUTOFFS[stimulus_cutoff_index]), FILTER_RIPPLE_DB,
                      FILTER_TRANSITION_HZ, MAINS_HZ, NOTCH_BANDWIDTH_HZ, len(paths), 1],
                     sort_keys=True)
    directory = CACHE_ROOT / REFERENCE
    directory.mkdir(parents=True, exist_ok=True)
    return directory / f"onset_trf_per_stimulus_{hashlib.sha1(key.encode()).hexdigest()[:12]}.npz"


def pool_stimuli(index):
    """`(H, n_trials, stimuli)`, where H is (n_stimuli, F, n_channels) at cutoff `index`.

    Within a stimulus, each trial is weighted by its analysed duration. That is the participant
    stage of the main pooling with the roles swapped. Each trial is a different listener, so it
    comes out as close to an equal-weight mean over listeners. The SD divisor is the same
    per-trial one used in the main pooling, at `TRF_BAND`.
    """
    totals: dict[str, np.ndarray] = {}
    weights: dict[str, float] = {}
    counts: dict[str, int] = {}
    for count, path in enumerate(paths, 1):
        with np.load(path) as saved:
            # Only the one cutoff, sliced before the reference change to keep it cheap.
            cross = saved[f"cross_spectrum_{ONSET_FEATURE_KEY}"][:, index:index + 1]
            stimulus_psd = saved[f"stimulus_psd_{ONSET_FEATURE_KEY}"][:, index:index + 1]
            names = [str(name) for name in saved["channel_names"]]
            psd = saved["eeg_psd"]
            stimulus = str(saved["stimulus"])
            duration = float(saved["analysed_duration"])

        cross, trial_channels = rereference_transfer_function(cross, names, REFERENCE)
        assert list(trial_channels) == channels, f"{path.stem}: unexpected channel axis"
        keep = [names.index(name) for name in trial_channels]
        sd = band_standard_deviation(psd[:, keep], TRF_RESPONSE)
        H = transfer_function_from_parts(cross, stimulus_psd, RIDGE)[:, 0] / sd

        if stimulus not in totals:
            totals[stimulus] = np.zeros(H.shape, complex)
            weights[stimulus], counts[stimulus] = 0.0, 0
        totals[stimulus] += duration * H
        weights[stimulus] += duration
        counts[stimulus] += 1
        if count % 100 == 0 or count == len(paths):
            print(f"  {count}/{len(paths)} trials", flush=True)

    stimuli = sorted(totals)
    fits = np.array([totals[name] / weights[name] for name in stimuli])
    return fits, np.array([counts[name] for name in stimuli]), stimuli


stimulus_cache = stimulus_cache_path()
if stimulus_cache.exists():
    with np.load(stimulus_cache) as saved:
        STIMULUS_FITS = saved["fits"]
        STIMULUS_TRIALS = saved["n_trials"]
        STIMULI = [str(name) for name in saved["stimuli"]]
    print(f"cached: {stimulus_cache.name}")
else:
    STIMULUS_FITS, STIMULUS_TRIALS, STIMULI = pool_stimuli(stimulus_cutoff_index)
    # complex64: the fits are float32 on disk to begin with, and this is 69 x F x 64.
    np.savez_compressed(stimulus_cache, fits=STIMULUS_FITS.astype(np.complex64),
                        n_trials=STIMULUS_TRIALS, stimuli=np.array(STIMULI))
    print(f"written: {stimulus_cache.name} ({stimulus_cache.stat().st_size / 1e6:.0f} MB)")


def stimulus_responses(band=PER_STIMULUS_BAND, channel=None):
    """One trace per stimulus, (n_stimuli, n_lags), read through `band`.

    `channel=None` gives the Hilbert envelope of every channel averaged over the 64. A channel
    name gives that channel's signed TRF, with no envelope taken.
    """
    fits = STIMULUS_FITS if channel is None else STIMULUS_FITS[..., channels.index(channel)]
    spectrum = fits * band_response(*band)[(None, slice(None)) + (None,) * (fits.ndim - 2)]
    trf = np.roll(np.fft.irfft(spectrum, n=nperseg, axis=1), nperseg // 2, axis=1)
    return np.abs(hilbert(trf, axis=1)).mean(-1) if channel is None else trf


print(f"{len(STIMULI)} stimuli at the {CUTOFFS[stimulus_cutoff_index]:g} Hz cutoff, "
      f"{STIMULUS_TRIALS.sum()} trials; listeners per stimulus {STIMULUS_TRIALS.min()}-"
      f"{STIMULUS_TRIALS.max()} (median {np.median(STIMULUS_TRIALS):g}), "
      f"{(STIMULUS_TRIALS < 3).sum()} stimuli with fewer than 3")

In [ ]:
# ===================== per-stimulus heatmap options ==========================
PER_STIMULUS_FIGSIZE = (6.27, 7.0)
PER_STIMULUS_LAGS = (-200.0, 800.0)      # where the onset response is; the rest is baseline
PER_STIMULUS_XTICKS = [-200, 0, 200, 400, 600, 800]
# What each row shows: "average" is the Hilbert envelope of every channel averaged over the 64,
# "channel" the signed TRF of PER_STIMULUS_CHANNEL alone, on a diverging scale.
PER_STIMULUS_MEASURE = "average"
PER_STIMULUS_CHANNEL = "FCz"
PER_STIMULUS_MIN_TRIALS = 1              # rows heard by fewer listeners than this are dropped
# How the rows are ordered: "latency" by each row's largest |value| inside PER_STIMULUS_PEAK_SEARCH,
# "trials" by listener count, "name" alphabetically. Sorting by latency makes a tidy-looking
# diagonal out of noise as well as out of signal, so judge it against "name".
PER_STIMULUS_SORT = "name"
PER_STIMULUS_PEAK_SEARCH = (0.0, 300.0)
# Each row divided by its own maximum. The onset feature's amplitude differs between stimuli and
# |H| scales inversely with it, so a shared scale would mostly show the regressor's amplitude.
# This is display scaling only; it does not touch the fits.
PER_STIMULUS_ROW_NORMALISE = True
PER_STIMULUS_CMAP = "magma"            # the envelope is positive: sequential
PER_STIMULUS_SIGNED_CMAP = IMAGE_CMAP   # a single channel's trace is signed: diverging
TRIALS_COLOUR = "#6b6b68"
# =============================================================================

if PER_STIMULUS_MEASURE not in ("average", "channel"):
    raise ValueError(f"PER_STIMULUS_MEASURE must be 'average' or 'channel', "
                     f"not {PER_STIMULUS_MEASURE!r}")
measure_channel = PER_STIMULUS_CHANNEL if PER_STIMULUS_MEASURE == "channel" else None
if measure_channel is not None and measure_channel not in channels:
    raise ValueError(f"{measure_channel} is not in the {REFERENCE} montage")
responses = stimulus_responses(channel=measure_channel)
signed = measure_channel is not None
kept = STIMULUS_TRIALS >= PER_STIMULUS_MIN_TRIALS
stimulus_shown = (times >= PER_STIMULUS_LAGS[0]) & (times <= PER_STIMULUS_LAGS[1])
stimulus_searched = (times >= PER_STIMULUS_PEAK_SEARCH[0]) & (times <= PER_STIMULUS_PEAK_SEARCH[1])
peak_latency = times[stimulus_searched][np.argmax(np.abs(responses[:, stimulus_searched]), axis=1)]

order = {"latency": np.argsort(peak_latency, kind="stable"),
         "trials": np.argsort(-STIMULUS_TRIALS, kind="stable"),
         "name": np.arange(len(STIMULI))}[PER_STIMULUS_SORT]
order = order[kept[order]]

plate = responses[order][:, stimulus_shown]
if PER_STIMULUS_ROW_NORMALISE:
    plate = plate / np.abs(plate).max(axis=1, keepdims=True)
# Symmetric about zero for a signed trace, from zero for an envelope.
plate_limit = 1.0 if PER_STIMULUS_ROW_NORMALISE else float(np.abs(plate).max())

stimulus_fig = plt.figure(figsize=PER_STIMULUS_FIGSIZE)
# The colourbar gets a column of its own, so (a) and (b) keep the same width and share a lag axis.
stimulus_grid = stimulus_fig.add_gridspec(2, 3, height_ratios=[4, 1.2],
                                          width_ratios=[1, 0.12, 0.025], hspace=0.18, wspace=0.06)
ax_heat = stimulus_fig.add_subplot(stimulus_grid[0, 0])
ax_count = stimulus_fig.add_subplot(stimulus_grid[0, 1], sharey=ax_heat)
ax_mean = stimulus_fig.add_subplot(stimulus_grid[1, 0], sharex=ax_heat)

rows = np.arange(len(order))
heat = ax_heat.pcolormesh(times[stimulus_shown], rows, plate,
                          cmap=PER_STIMULUS_SIGNED_CMAP if signed else PER_STIMULUS_CMAP,
                          shading="nearest", vmin=-plate_limit if signed else 0,
                          vmax=plate_limit, rasterized=True)
ax_heat.axvline(0, color=INK if signed else "w", lw=0.8, alpha=0.6)
ax_heat.set_ylim(len(order) - 0.5, -0.5)
ax_heat.set_yticks(rows)
ax_heat.set_yticklabels([STIMULI[i].replace("audiobook_", "A").replace("podcast_", "P")
                         for i in order], fontsize=4.5)
ax_heat.set_ylabel("Stimulus")
ax_heat.tick_params(axis="x", labelbottom=False)

# Listener count beside each row: how much one row can be trusted.
ax_count.barh(rows, STIMULUS_TRIALS[order], color=TRIALS_COLOUR, height=0.8)
ax_count.set_xscale("log")
ax_count.set_xticks([1, 10, 50])
ax_count.set_xticklabels(["1", "10", "50"], fontsize=SMALL_FONTSIZE)
ax_count.set_title("Listeners", fontsize=SMALL_FONTSIZE, pad=3)
ax_count.tick_params(axis="y", left=False, labelleft=False)
bare(ax_count)

colorbar = plt.colorbar(heat, cax=stimulus_fig.add_subplot(stimulus_grid[0, 2]))
measure_label = "Channel-averaged envelope" if measure_channel is None else measure_channel
colorbar.set_label(f"{measure_label}, $\\div$ row's largest |value|" if PER_STIMULUS_ROW_NORMALISE
                   else f"{measure_label} (a.u.)",
                   fontsize=SMALL_FONTSIZE)
if PER_STIMULUS_ROW_NORMALISE:
    colorbar.set_ticks([-1, 0, 1] if signed else [0, 0.5, 1])
colorbar.ax.tick_params(labelsize=TICK_FONTSIZE)

# (b) the grand-average TRF: the participant-pooled fit the figures above draw, at the same cutoff
# and band, every channel coloured by position. The quantity each row of (a) shows is drawn over
# it in black: the channel-averaged envelope, or the chosen channel's signed trace.
grand_trf, grand_envelope = onset_response(stimulus_cutoff_index, PER_STIMULUS_BAND)
for index in range(grand_trf.shape[1]):
    ax_mean.plot(times[stimulus_shown], grand_trf[stimulus_shown, index],
                 color=CHANNEL_COLOURS[index], lw=0.5, alpha=0.45, zorder=2)
if measure_channel is None:
    ax_mean.plot(times[stimulus_shown], grand_envelope[stimulus_shown].mean(-1), color=INK,
                 lw=1.2, zorder=5, label="Channel-averaged envelope")
else:
    column = channels.index(measure_channel)
    ax_mean.plot(times[stimulus_shown], grand_trf[stimulus_shown, column], color=INK, lw=1.2,
                 zorder=5, label=measure_channel)
ax_mean.axvline(0, color=HAIRLINE, lw=0.8, zorder=0)
ax_mean.set_xlim(*PER_STIMULUS_LAGS)
ax_mean.set_xticks(PER_STIMULUS_XTICKS)
ax_mean.set_xlabel("Lag (ms)")
ax_mean.axhline(0, color=HAIRLINE, lw=0.8, zorder=0)
ax_mean.set_ylabel("Activity (a.u.)")
grand_limit = float(np.abs(grand_trf[stimulus_shown]).max()) * SINGLE_HEADROOM
ax_mean.set_ylim(-grand_limit, grand_limit)
ax_mean.legend(fontsize=SMALL_FONTSIZE, frameon=False, loc="upper right")
plot_sensor_key(ax_mean.inset_axes([0.0, 0.55, 0.09, 0.45]))
bare(ax_mean)

for ax, label in [(ax_heat, "a)"), (ax_mean, "b)")]:
    ax.text(-0.12, 1.02, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = (f"onset_trf_per_stimulus_{measure_channel or 'chanavg'}"
        f"_{PER_STIMULUS_BAND[0]:g}-{PER_STIMULUS_BAND[1]:g}Hz"
        f"_cut{CUTOFFS[stimulus_cutoff_index]:g}_{REFERENCE}")
stimulus_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
stimulus_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

print(f"{measure_label}, {PER_STIMULUS_BAND[0]:g}-{PER_STIMULUS_BAND[1]:g} Hz EEG, "
      f"{CUTOFFS[stimulus_cutoff_index]:g} Hz peakRate cutoff, {kept.sum()}/{len(STIMULI)} "
      f"stimuli with >= {PER_STIMULUS_MIN_TRIALS} listeners, sorted by {PER_STIMULUS_SORT}")
for low, high in [(1, 2), (3, 9), (10, None)]:
    group = (STIMULUS_TRIALS >= low) & ((STIMULUS_TRIALS <= high) if high else True)
    q = np.percentile(peak_latency[group], [10, 50, 90])
    print(f"  {low}{'-' + str(high) if high else '+'} listeners ({group.sum():2d} stimuli): "
          f"peak latency median {q[1]:.0f} ms, 10-90% {q[0]:.0f}-{q[2]:.0f} ms")

In [ ]:
responses[order][:, stimulus_shown]

In [ ]:
fig, axs = plt.subplots(10, 1, figsize=(6.27, 10), sharex=True, constrained_layout=True)
for index, ax in enumerate(axs):
    ax.plot(times[stimulus_shown], responses[index*4], color=INK, lw=1.2)